In [7]:
import timesformer.utils.checkpoint as cu
from timesformer.models import build_model
import timesformer.models.optimizer as optim
from timesformer.utils.parser import load_config, parse_args

In [19]:
from types import SimpleNamespace

# Simulate the parsed arguments using SimpleNamespace
args = SimpleNamespace(
    shard_id=0,  # Default value for --shard_id
    num_shards=1,  # Default value for --num_shards
    init_method="tcp://localhost:9999",  # Default value for --init_method
    cfg_file="/home/yygx/UNC_Research/pkgs_baselines/TimeSformer/configs/FrankaHuman/TimeSformer_divST_8x32_224.yaml",  # Default value for --cfg
    opts=None  # Default value for additional options
)

# Optionally, print or inspect the args object
print(args)

if args.num_shards > 1:
   args.output_dir = str(args.job_dir)
cfg = load_config(args)
model = build_model(cfg)
optimizer = optim.construct_optimizer(model, cfg)
start_epoch = cu.load_train_checkpoint(cfg, model, optimizer)

namespace(cfg_file='/home/yygx/UNC_Research/pkgs_baselines/TimeSformer/configs/FrankaHuman/TimeSformer_divST_8x32_224.yaml', init_method='tcp://localhost:9999', num_shards=1, opts=None, shard_id=0)


In [22]:
model.model.head

Linear(in_features=768, out_features=400, bias=True)

In [34]:
import torch.nn as nn
import torch
c = torch.randn(3, 9, 3, 112, 112).cuda()
# dummy_video = torch.randn(2, 3, 8, 224, 224).cuda()
dummy_video = c.permute(0, 2, 1, 3, 4)
model.model.head = nn.Identity()
output = model(dummy_video)

In [35]:
output.size()

torch.Size([3, 768])

In [25]:
model

vit_base_patch16_224(
  (model): VisionTransformer(
    (dropout): Dropout(p=0.0, inplace=False)
    (patch_embed): PatchEmbed(
      (proj): Conv2d(3, 768, kernel_size=(16, 16), stride=(16, 16))
    )
    (pos_drop): Dropout(p=0.0, inplace=False)
    (time_drop): Dropout(p=0.0, inplace=False)
    (blocks): ModuleList(
      (0): Block(
        (norm1): LayerNorm((768,), eps=1e-06, elementwise_affine=True)
        (attn): Attention(
          (qkv): Linear(in_features=768, out_features=2304, bias=True)
          (proj): Linear(in_features=768, out_features=768, bias=True)
          (proj_drop): Dropout(p=0.0, inplace=False)
          (attn_drop): Dropout(p=0.0, inplace=False)
        )
        (temporal_norm1): LayerNorm((768,), eps=1e-06, elementwise_affine=True)
        (temporal_attn): Attention(
          (qkv): Linear(in_features=768, out_features=2304, bias=True)
          (proj): Linear(in_features=768, out_features=768, bias=True)
          (proj_drop): Dropout(p=0.0, inplace=F

In [ ]:
import torch
from timesformer.models.vit import TimeSformer
import torch.nn as nn

model = TimeSformer(img_size=224, num_classes=400, num_frames=8, attention_type='divided_space_time',  pretrained_model='/home/yygx/UNC_Research/pkgs_baselines/TimeSformer/checkpoints/checkpoint_epoch_00015.pyth')

model.model.head = nn.Identity()
# torch.Size([54, 3, 9, 224, 224])
dummy_video = torch.randn(54, 3, 9, 224, 224) # (batch x channels x frames x height x width)

pred = model(dummy_video,) # (2, 400)

In [37]:
pred.shape

torch.Size([2, 768])